In [1]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta, timezone

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("AAPL_Streaming_Analytics").getOrCreate()

26/08/11 17:40:27 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
import os

# Set this before running: export POLYGON_API_KEY="your-key-here"
API_KEY = os.environ["POLYGON_API_KEY"]
BASE_URL = "https://api.massive.com"
TICKER = "AAPL"


def get_histdata_polygon(ticker, start_date, end_date, timespan="minute", multiplier=1):
    sd = start_date.strftime("%Y-%m-%d")
    ed = end_date.strftime("%Y-%m-%d")

    url = (f"{BASE_URL}/v2/aggs/ticker/{ticker}/range/{multiplier}/{timespan}/"
           f"{sd}/{ed}?adjusted=true&sort=asc&limit=50000&apiKey={API_KEY}")
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        if "results" not in data or len(data["results"]) == 0:
            return pd.DataFrame()
        df = pd.DataFrame(data["results"])
        df["timestamp"] = pd.to_datetime(df["t"], unit="ms", utc=True)
        df = df[["timestamp", "o", "h", "l", "c", "v"]]
        df.columns = ["timestamp", "open", "high", "low", "close", "volume"]
        df = df[(df["timestamp"] >= start_date) & (df["timestamp"] <= end_date)]
        return df
    elif response.status_code == 403:
        print("Error 403: Forbidden. Check your API key and subscription plan.")
        return pd.DataFrame()
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return pd.DataFrame()

In [3]:
def previous_business_day(d):
    d -= timedelta(days=1)
    while d.weekday() >= 5:
        d -= timedelta(days=1)
    return d


today_utc = datetime.now(timezone.utc).date()
anchor_day = previous_business_day(datetime.combine(today_utc, datetime.min.time()))
ANCHOR_END = datetime(anchor_day.year, anchor_day.month, anchor_day.day, 14, 0, 0, tzinfo=timezone.utc)

print(f"Replaying AAPL data anchored at {ANCHOR_END.isoformat()} (previous business day)")

Replaying AAPL data anchored at 2026-08-10T14:00:00+00:00 (previous business day)


In [4]:
raw_schema = StructType([
    StructField("Stock Name", StringType(), True),
    StructField("UTC Timestamp", TimestampType(), True),
    StructField("c", DoubleType(), True),
    StructField("l", DoubleType(), True),
    StructField("h", DoubleType(), True),
    StructField("o", DoubleType(), True),
    StructField("v", DoubleType(), True),
])

ma_schema = StructType([
    StructField("Datetime", TimestampType(), True),
    StructField("c_MA", DoubleType(), True),
    StructField("l_MA", DoubleType(), True),
    StructField("h_MA", DoubleType(), True),
    StructField("o_MA", DoubleType(), True),
    StructField("v_MA", DoubleType(), True),
])

main_df = spark.createDataFrame([], schema=raw_schema)
ma_df = spark.createDataFrame([], schema=ma_schema)

print("Schemas and empty dataframes created.")

Schemas and empty dataframes created.


In [5]:
NUM_PULLS = 7
SLEEP_SECONDS = 300  # 5 minutes

for pull_num in range(NUM_PULLS):
    sim_now = ANCHOR_END + timedelta(minutes=5 * pull_num)
    window_start = sim_now - timedelta(hours=1)
    window_end = sim_now

    print(f"\n--- Pull {pull_num + 1}/{NUM_PULLS} | simulated 'now' = {sim_now} ---")
    print(f"Requesting window: {window_start} -> {window_end}")

    pdf = get_histdata_polygon(TICKER, window_start, window_end, timespan="minute", multiplier=1)

    if pdf.empty:
        print("No data returned for this pull, skipping merge.")
    else:
        pdf["Stock Name"] = TICKER
        pdf = pdf.rename(columns={
            "timestamp": "UTC Timestamp", "close": "c",
            "low": "l", "high": "h", "open": "o", "volume": "v"
        })
        pdf = pdf[["Stock Name", "UTC Timestamp", "c", "l", "h", "o", "v"]]

        new_spark_df = spark.createDataFrame(pdf, schema=raw_schema)

        unchanged_existing = main_df.join(
            new_spark_df.select("UTC Timestamp"),
            on="UTC Timestamp",
            how="left_anti"
        )
        main_df = unchanged_existing.unionByName(new_spark_df).orderBy("UTC Timestamp")

        print(f"Main dataframe row count after merge: {main_df.count()}")

    ma_window_start = window_end - timedelta(minutes=30)
    window_slice = main_df.filter(
        (F.col("UTC Timestamp") > ma_window_start) & (F.col("UTC Timestamp") <= window_end)
    )

    if window_slice.count() > 0:
        agg_row = window_slice.agg(
            F.avg("c").alias("c_MA"),
            F.avg("l").alias("l_MA"),
            F.avg("h").alias("h_MA"),
            F.avg("o").alias("o_MA"),
            F.avg("v").alias("v_MA"),
        ).collect()[0]

        new_ma_row = spark.createDataFrame(
            [(window_end, agg_row["c_MA"], agg_row["l_MA"], agg_row["h_MA"], agg_row["o_MA"], agg_row["v_MA"])],
            schema=ma_schema
        )
        ma_df = ma_df.unionByName(new_ma_row)
        print("30-min moving averages updated.")
    else:
        print("Not enough data yet for a moving-average window.")

    if pull_num < NUM_PULLS - 1:
        print(f"Sleeping {SLEEP_SECONDS} seconds until next pull...")
        time.sleep(SLEEP_SECONDS)

print("\n=== Streaming simulation complete ===")


--- Pull 1/7 | simulated 'now' = 2026-08-10 14:00:00+00:00 ---
Requesting window: 2026-08-10 13:00:00+00:00 -> 2026-08-10 14:00:00+00:00


Main dataframe row count after merge: 61
30-min moving averages updated.
Sleeping 300 seconds until next pull...

--- Pull 2/7 | simulated 'now' = 2026-08-10 14:05:00+00:00 ---
Requesting window: 2026-08-10 13:05:00+00:00 -> 2026-08-10 14:05:00+00:00
Main dataframe row count after merge: 66
30-min moving averages updated.
Sleeping 300 seconds until next pull...

--- Pull 3/7 | simulated 'now' = 2026-08-10 14:10:00+00:00 ---
Requesting window: 2026-08-10 13:10:00+00:00 -> 2026-08-10 14:10:00+00:00


Main dataframe row count after merge: 71
30-min moving averages updated.
Sleeping 300 seconds until next pull...

--- Pull 4/7 | simulated 'now' = 2026-08-10 14:15:00+00:00 ---
Requesting window: 2026-08-10 13:15:00+00:00 -> 2026-08-10 14:15:00+00:00
Main dataframe row count after merge: 76
30-min moving averages updated.
Sleeping 300 seconds until next pull...

--- Pull 5/7 | simulated 'now' = 2026-08-10 14:20:00+00:00 ---
Requesting window: 2026-08-10 13:20:00+00:00 -> 2026-08-10 14:20:00+00:00


Main dataframe row count after merge: 81


30-min moving averages updated.
Sleeping 300 seconds until next pull...

--- Pull 6/7 | simulated 'now' = 2026-08-10 14:25:00+00:00 ---
Requesting window: 2026-08-10 13:25:00+00:00 -> 2026-08-10 14:25:00+00:00


Main dataframe row count after merge: 86


30-min moving averages updated.
Sleeping 300 seconds until next pull...

--- Pull 7/7 | simulated 'now' = 2026-08-10 14:30:00+00:00 ---
Requesting window: 2026-08-10 13:30:00+00:00 -> 2026-08-10 14:30:00+00:00


Main dataframe row count after merge: 91


30-min moving averages updated.

=== Streaming simulation complete ===


In [6]:
print("Final streamed dataframe (main_df):")
main_df.orderBy("UTC Timestamp").show(100, truncate=False)

print("\nFinal moving averages dataframe (ma_df):")
ma_df.orderBy("Datetime").show(100, truncate=False)

Final streamed dataframe (main_df):


+-------------------+----------+--------+--------+--------+----------+--------------+
|UTC Timestamp      |Stock Name|c       |l       |h       |o         |v             |
+-------------------+----------+--------+--------+--------+----------+--------------+
|2026-08-10 13:00:00|AAPL      |309.18  |309.05  |309.1986|309.05    |11103.031054  |
|2026-08-10 13:01:00|AAPL      |309.2098|309.05  |309.2098|309.07    |5034.143329   |
|2026-08-10 13:02:00|AAPL      |308.94  |308.94  |309.19  |309.1     |16814.147142  |
|2026-08-10 13:03:00|AAPL      |308.96  |308.9   |309.06  |308.9     |4114.479768   |
|2026-08-10 13:04:00|AAPL      |309.0   |308.91  |309.0   |308.94    |3822.0        |
|2026-08-10 13:05:00|AAPL      |309.05  |309.0   |309.05  |309.0     |2072.31935    |
|2026-08-10 13:06:00|AAPL      |308.96  |308.94  |309.07  |309.02    |7473.551864   |
|2026-08-10 13:07:00|AAPL      |309.0   |308.9   |309.05  |308.9252  |4563.316874   |
|2026-08-10 13:08:00|AAPL      |309.0   |308.97  |309.

+-------------------+------------------+------------------+------------------+------------------+------------------+
|Datetime           |c_MA              |l_MA              |h_MA              |o_MA              |v_MA              |
+-------------------+------------------+------------------+------------------+------------------+------------------+
|2026-08-10 14:00:00|306.2433          |305.8976          |306.57126         |306.25385339999997|221415.06695810007|
|2026-08-10 14:05:00|306.4509300000001 |306.1731          |306.6772166666667 |306.4171867333333 |176531.94647126665|
|2026-08-10 14:10:00|306.5968033333334 |306.37422000000004|306.8162166666668 |306.5822166666667 |161265.8644214667 |
|2026-08-10 14:15:00|306.63138         |306.4222200000001 |306.84039333333334|306.6180166666667 |144124.55847139997|
|2026-08-10 14:20:00|306.69006666666667|306.51622333333336|306.87672333333325|306.67735000000005|127429.49182433335|
|2026-08-10 14:25:00|306.79007         |306.62457333333333|306.9